In [0]:
print("-----GIVEN DATAFRAME-----\n\n\n")


data = [
    (1111, "2021-01-15", 10),
    (1111, "2021-01-16", 15),
    (1111, "2021-01-17", 30),
    (1112, "2021-01-15", 10),
    (1112, "2021-01-15", 20),
    (1112, "2021-01-15", 30)
]

columns = ["sensorid", "timestamp", "values"]

df = spark.createDataFrame(data, columns)
df.show()

print("-----REQUIRED DATAFRAME-----\n\n\n")

data1 = [
    (1111, "2021-01-15", 5),
    (1111, "2021-01-16", 15),
    (1112, "2021-01-15", 10),
    (1112, "2021-01-15", 10)
]

columns1 = ["sensorid", "timestamp", "values"]

df1 = spark.createDataFrame(data1, columns1)
df1.show()


print("-----DSL SOLUTION-----\n\n\n")

from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col


windowdf = Window.partitionBy("sensorid").orderBy("timestamp")

leaddf = df.withColumn("lead", lead(col("values"),1).over(windowdf))\
    .withColumn("change", col("lead")-col("values"))\
        .drop("lead","values")\
            .withColumnRenamed("change","values")\
                .dropna()


leaddf.show()


print("-----SQL SOLUTION-----\n\n\n")


df.createOrReplaceTempView("df_view")

spark.sql("""
          select sensorid,timestamp, change as values from
           (select sensorid, timestamp,values,
          lead(values,1) OVER (partition by sensorid order by timestamp) as lead,
          lead - values as change
          from df_view)
          where change is not null
          """).show()



